In [ ]:
#pip install openchord

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import itertools
import re
import openchord as ocd
# https://github.com/pke1029/open-chord/tree/main

In [ ]:
pd.set_option("display.max_columns", None)

# Loading the full Victim Form file

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

In [ ]:
essential_cols = ["wave","global_person_id"]

why_not_report_cols = [
    "xycopno23a",
    "xycopno23b",
    "xycopno23c",
    "xycopno23d",
    "xycopno23e",
    "xycopno23f",
    "xycopno23g",
    "xycopno23h",
    "xycopno23i",
    "xycopno23j",
    "xycopno23k",
    "xycopno23l",
    "xycopno23m",
    "xycopno23n",
    "xycopno23o",
    "xycopno23p",
    "xycopno23q",
    "xycopno23r",
    "xycopno23s",
    "xycopno23t",
    "xycopno23u",
    "xycopno23v",
    "xycopno23w",
    "xycopno23x",
]

why_not_report_cols += [
    new_col
    for col in why_not_report_cols.copy()
    for new_col in [
        col[1:],                  # ycopno23a
        col.replace("23", "2"),  # xycopno2a
        col[1:].replace("23", "2"),  # ycopno2a
    ]
]

vf_cols = pd.read_excel(f_root / "data/csew/vars_match/csew_vf_data_dictionary.xlsx")
vf_cols_keep = vf_cols.loc[vf_cols["priority"].eq("priority"),"var"].tolist()
cols_to_load = list(set(essential_cols + vf_cols_keep + why_not_report_cols))

In [ ]:
csew_vf = pd.read_csv(
    f_root / "data/csew/merged/Post2011_vf.tab",
    sep="\t",
    usecols=lambda c: c in cols_to_load,
    dtype=str,
    low_memory=True,
    na_values=["<NA>", "NaN"]
)

In [ ]:
# Convert wave from object/string to numeric year
csew_vf["year"] = pd.to_numeric(csew_vf["wave"],errors="coerce").astype("Int64")

In [ ]:
csew_vf.shape

In [ ]:
csew_vf.columns

# VF variables adjustments

In [ ]:
csew_vf_amended = csew_vf.copy()

In [ ]:
# Keep only columns that exist in the DataFrame
available_cols = [
    col for col in why_not_report_cols
    if col in csew_vf_amended.columns
]

# Group columns by their final letter
cols_by_letter = {}

for col in available_cols:
    match = re.search(r"([a-z])$", col)

    if match:
        letter = match.group(1)
        cols_by_letter.setdefault(letter, []).append(col)

# Combine each group into ycopno_{letter}
for letter, cols in cols_by_letter.items():
    source = csew_vf_amended[cols].apply(
        pd.to_numeric,
        errors="coerce",
    )

    csew_vf_amended[f"ycopno_{letter}"] = (
        source.eq(1)
        .any(axis=1)
        .astype("boolean")
        .mask(source.isna().all(axis=1))
    )

# Drop all original columns
csew_vf_amended.drop(
    columns=available_cols,
    inplace=True,
)

In [ ]:
csew_vf_amended

In [ ]:
csew_vf_amended.info()

## Offender - survivor relationship

In [ ]:
# harmonising and simplifying relationship-to-offender columns
# Convert *offrel3 codes to the equivalent *offrel4 codes
offrel3_to_offrel4 = {
    "1": "1",    # Husband/wife/partner
    "2": "2",    # Son/daughter (in law)
    "3": "3",    # Other household member
    "4": "4",    # Current boyfriend/girlfriend
    "5": "5",    # Former husband/wife/partner
    "6": "6",    # Former boyfriend/girlfriend
    "7": "7",    # Other relative
    "8": "8",    # Workmate/colleague
    "9": "9",    # Client/member of public through work
    "10": "10",  # Friend/acquaintance
    "11": "12",  # Neighbour
    "12": "13",  # Young people from local area
    "13": "14",  # Tradesman/builder/contractor
    "14": "15",  # Ex-partner of someone else in household
    "15": "16",  # Other
    "98": np.nan,
    "99": np.nan,
}

# Group the harmonised relationship codes
offrel_group_map = {
    # Current or former intimate partner
    "1": "current or former intimate partner",
    "4": "current or former intimate partner",
    "5": "current or former intimate partner",
    "6": "current or former intimate partner",

    # Relative or household family relationship
    "2": "relative or acquaintance",    "3": "relative or acquaintance",
    "7": "relative or acquaintance",

    # Other known person
    "8": "relative or acquaintance",   # Workmate/colleague
    "9": "relative or acquaintance",   # Client/public through work
    "10": "relative or acquaintance",  # Friend/acquaintance
    "12": "relative or acquaintance",  # Neighbour
    "13": "relative or acquaintance",  # Young people from local area
    "14": "relative or acquaintance",  # Tradesman/contractor
    "15": "relative or acquaintance",  # Ex-partner of someone else in household
    "16": "relative or acquaintance",  # Other known/unspecified relationship

    # No relationship / stranger
    "11": "other/stranger",
}


def harmonise_two_columns(
    df,
    old_col,
    new_col,
    output_col,
):
    old_recoded = df[old_col].map(offrel3_to_offrel4)

    combined = df[new_col].fillna(old_recoded)

    df[output_col] = combined.map(offrel_group_map)

    df.drop(
        columns=[old_col, new_col],
        inplace=True,
    )


# offrel3 / offrel4 -> offrel
harmonise_two_columns(
    csew_vf_amended,
    old_col="offrel3",
    new_col="offrel4",
    output_col="offrel",
)

# foffrel3 / foffrel4 -> foffrel
harmonise_two_columns(
    csew_vf_amended,
    old_col="foffrel3",
    new_col="foffrel4",
    output_col="foffrel",
)

# Merge foffrel into offrel, using foffrel only where offrel is missing
csew_vf_amended["offrel"] = (
    csew_vf_amended["offrel"]
    .fillna(csew_vf_amended["foffrel"])
)

# Drop foffrel after merging
csew_vf_amended.drop(
    columns=["foffrel"],
    inplace=True,
)

print(csew_vf_amended["offrel"].value_counts(dropna=False))

In [ ]:
# Variables for which you want null-count bar plots
variables_to_plot = [ "offrel"]

# Count nulls for each selected variable within each year
null_counts_by_year = (csew_vf_amended.groupby("year", dropna=False)[variables_to_plot].agg(lambda column: column.isna().sum()))

# Create one bar plot per variable
for variable in variables_to_plot:
    plot_data = null_counts_by_year[variable]

    plt.figure(figsize=(8, 3))
    plot_data.plot(kind="bar")

    plt.title(f"NULL {variable} by year")
    plt.xlabel("Year")
    plt.ylabel("Number of NULLs")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Recoding the rest of the useful columns

In [ ]:
emotreac_map = {
    "1": "Emotional after",
    "2": "Not emotional after",
    "9": np.nan,"8": np.nan, "0": np.nan}
csew_vf_amended["emotreac"] = csew_vf_amended["emotreac"].map(emotreac_map)

respinj_map = {
    "1": "Was injured",
    "0": "Was not injured",
    "9": np.nan,"8": np.nan, "0": np.nan}
csew_vf_amended["respinj"] = csew_vf_amended["respinj"].map(respinj_map)

att_map = {
    "0": "No medical attention",
    "1": "Received medical attention"}
csew_vf_amended["att"] = csew_vf_amended["att"].map(att_map)

pincid_map = {
    "1": "series_incident",
    "2": "single_incident",
    "9": np.nan,"8": np.nan, "0": np.nan}
csew_vf_amended["pincid"] = csew_vf_amended["pincid"].map(pincid_map)

crime_map = {
    "1": "Consider it crime or wrong",
    "2": "Consider it crime or wrong",
    "3": "Something that happens",
    "9": np.nan,"8": np.nan, "0": np.nan}
csew_vf_amended["crime"] = csew_vf_amended["crime"].map(crime_map)


csew_vf_amended["nseries"] = pd.to_numeric(
    csew_vf_amended["nseries"], errors="coerce"
).astype("Int64")

csew_vf_amended["inc_num_uncapped"] = pd.cut(
    csew_vf_amended["nseries"],
    bins=[-1, 0, 10, 20, float("inf")],
    labels=[
        "single incident",
        "2 to 10 incidents in series",
        "10 to 20 incidents in series",
        "over 20 incidents in series"
    ]
)

In [ ]:
ycopno_true_labels = {
    "ycopno_a": "Unreported because Private/personal/family matter",
    "ycopno_b": "Unreported because Dealt with matter myself/ourselves",
    "ycopno_c": "Unreported because Reported to other authorities",
    "ycopno_d": "Unreported because Dislike of police",
    "ycopno_e": "Unreported because Fear of police",
    "ycopno_f": "Unreported because Lack of trust in the police",
    "ycopno_g": "Unreported because Fear of reprisal by offenders or making matters worse",
    "ycopno_h": "Unreported because Police could have done nothing",
    "ycopno_i": "Unreported because Police would not have bothered or been interested",
    "ycopno_j": "Unreported because It was inconvenient or too much trouble",
    "ycopno_k": "Unreported because There was no loss or damage",
    "ycopno_l": "Unreported because The attempted offence was unsuccessful",
    "ycopno_m": "Unreported because It was too trivial or not worth reporting",
    "ycopno_n": "Unreported because Previous negative experience of the police or courts",
    "ycopno_o": "Unreported because Family or friends had a negative experience of the police or courts",
    "ycopno_p": "Unreported because It was a common event or just something that happens",
    "ycopno_q": "Unreported because It was something that happens as part of my job",
    "ycopno_r": "Unreported because It was partly my, a relative's, or a friend's fault",
    "ycopno_s": "Unreported because The offender was not responsible for their actions",
    "ycopno_t": "Unreported because Someone else had already reported it",
    "ycopno_u": "Unreported because Tried to report it but could not contact the police",
    "ycopno_v": "Unreported for another reason",
    "ycopno_w": "Unreported reason unknown",
    "ycopno_x": "Unreported reason refused",
}

for col, true_label in ycopno_true_labels.items():
    if col in csew_vf_amended.columns:
        csew_vf_amended[col] = (
            csew_vf_amended[col]
            .astype("boolean")
            .map({
                True: true_label,
                False: "False",
            })
            .astype("string")
        )

In [ ]:
csew_vf["nseries"].value_counts()

In [ ]:
csew_vf_amended

## Recoding non-reporting

In [ ]:
# Recode both old and new versions to the same labels
copsknow_map = {
    "1": 1,
    "2": 2,
    "9": np.nan,"8": np.nan,"0": np.nan}

copsknow_recoded = csew_vf_amended["copsknow"].map(copsknow_map)
fcopsknow_recoded = csew_vf_amended["fcopsknow"].map(copsknow_map)
csew_vf_amended["copsknow_new"] = (copsknow_recoded.fillna(fcopsknow_recoded))
csew_vf_amended.drop(columns=["fcopsknow", "copsknow"],inplace=True)

print(csew_vf_amended["copsknow_new"].value_counts(dropna=False))

csew_vf_amended.loc[csew_vf_amended["howcopk"].isin(["1"]),"resp_told_police"] = 1
csew_vf_amended.loc[csew_vf_amended["howcopk"].isin(["2","3","4"]),"reported_discard"] = True

## Applying VAWG filters

In [ ]:
csew_vf_amended = csew_vf_amended[csew_vf_amended["offence"] != "96"].copy() #filering out invalid victim forms
csew_vf_amended = csew_vf_amended[csew_vf_amended["offence"] != "2"].copy() #filering out invalid victim forms

In [ ]:
offence_codes_sexual = [
    "31",  # Rape
    "32",  # Serious wounding with sexual motive
    "33",  # Other wounding with sexual motive
    "34",  # Attempted rape
    "35",  # Indecent assault
    "39",  # Sexual offence outside the surveys coverage
    "92",  # Sexual threat
]

In [ ]:
csew_vf_amended["violgrp"].value_counts()

In [ ]:
# those victim forms that have coding that matches official sexual offences categories
csew_vf_amended["vawg_official_codes"] = (csew_vf_amended["offence"].isin(offence_codes_sexual).astype("boolean").mask(csew_vf_amended["offence"].isna()))

# if there were any other crimes at all, where there were sexual threats
# or sexual violence was used as force
sex_force_threat_cols = ["whatfo2f", "whatfo4f", "whatfo2g", "whatfo4g","whatfo2h", "whatfo4h", "whthre3c", "whthre4c","v712", "fv712",]
sex_force_data = csew_vf_amended[sex_force_threat_cols]
csew_vf_amended["vawg_sex_force_threats"] = (
    sex_force_data.eq("1").any(axis=1).astype("boolean").mask(sex_force_data.isna().all(axis=1)))

# any domestic or partner violence
# latter if relationship history was given as a reason for violence
whyhap_cols = ["whyhap3d", "whyhap4d", "whyhap3b", "whyhap4b"]
violgrp_yes = csew_vf_amended["violgrp"].eq("1")
partner = csew_vf_amended["offrel"].eq("current or former intimate partner")

whyhap_yes = csew_vf_amended[whyhap_cols].eq("1").any(axis=1)
domestic_sources = csew_vf_amended[["violgrp", "offrel"] + whyhap_cols]

csew_vf_amended["vawg_domestic"] = ((violgrp_yes | (partner & whyhap_yes)).astype("boolean").mask(domestic_sources.isna().all(axis=1)))

#vawg harassment
harassment = ["harasmotx", "fharasmotx"]
verbal_abuse = ["whatfo4i", "whatfo2i"]
source_cols = harassment + verbal_abuse

csew_vf_amended["vawg_harasm"] = (
    (csew_vf_amended["ofsex"].eq("1")
        & csew_vf_amended[source_cols].eq("1").any(axis=1)
    ).astype("boolean").mask(
        csew_vf_amended["ofsex"].isna()
        | csew_vf_amended[source_cols].isna().all(axis=1)
    ))  

In [ ]:
variables_to_plot = ["vawg_official_codes", "vawg_sex_force_threats", "vawg_domestic", "vawg_harasm"]
null_counts_by_year = (csew_vf_amended.groupby("year", dropna=False)[variables_to_plot].agg(lambda column: column.isna().sum()))

# Create one bar plot per variable
for variable in variables_to_plot:
    plot_data = null_counts_by_year[variable]

    plt.figure(figsize=(8, 2))
    plot_data.plot(kind="bar")

    plt.title(f"NULL {variable} by year")
    plt.xlabel("Year")
    plt.ylabel("Number of NULLs")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
cols_to_drop = list(dict.fromkeys(source_cols + ["violgrp", "wherhapp", "victarea", "nseries", "numinc", "Numoff2", "whatfo3i", "cyber"] + whyhap_cols 
                                  # + sex_force_threat_cols
                                 ))
csew_vf_amended.drop(columns=cols_to_drop,inplace=True,errors="ignore")
csew_vf_amended.head(3)

# Export

In [ ]:
csew_vf_amended["whatfo4f"].value_counts()

In [ ]:
csew_vf_amended.to_csv(
    f_root / "data/csew/merged/vf_lookup_TEST.csv"
)

In [ ]:
csew_vf_amended["att"].value_counts()

In [ ]:
csew_vf_amended.to_csv(
    f_root / "data/csew/merged/vf_lookup.csv"
)

In [ ]:
csew_vf_amended